ARTI308 - Machine Learning

# Lab 4: Data Quality Assessment & Preprocessing

In real-world machine learning projects, data is often:
- Incomplete (missing values)
- Noisy (outliers or random errors)
- Inconsistent (wrong formats, mixed units)

Before building any machine learning model, we must clean and prepare the data properly.

In this lab, we will apply practical preprocessing techniques step by step using a **Password Strength dataset**.

In [ ]:
# Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

## 1. Load Dataset

In [ ]:
pd.set_option("display.max_columns", None)

df = pd.read_csv("data.csv", on_bad_lines='skip')
df.head(10)

The dataset contains **password** entries and their corresponding **strength** labels:
- `0` = Weak
- `1` = Medium
- `2` = Strong

To enable numerical analysis, we will extract useful features from each password.

In [ ]:
# Extract numerical features from passwords
df['length']      = df['password'].astype(str).str.len()
df['num_digits']  = df['password'].astype(str).str.count(r'[0-9]')
df['num_upper']   = df['password'].astype(str).str.count(r'[A-Z]')
df['num_special'] = df['password'].astype(str).str.count(r'[^a-zA-Z0-9]')

df.head(10)

## 2. Data Quality Assessment
### 2.1 Check Data Types
Data types must match the real meaning of each column.
For example:
- `password` should be a string (object)
- `strength` should be numeric (int)
- `length`, `num_digits`, `num_upper`, `num_special` should be numeric

In [ ]:
df.dtypes

### 2.2 Convert Incorrect Data Types
We ensure that all numerical feature columns are stored as the correct numeric type.

In [ ]:
df['length']      = pd.to_numeric(df['length'],      errors='coerce')
df['num_digits']  = pd.to_numeric(df['num_digits'],  errors='coerce')
df['num_upper']   = pd.to_numeric(df['num_upper'],   errors='coerce')
df['num_special'] = pd.to_numeric(df['num_special'], errors='coerce')
df['strength']    = pd.to_numeric(df['strength'],    errors='coerce')

df.dtypes

All features are now stored as numeric types, allowing us to perform statistical calculations correctly.

## 3. Handling Missing Values
### 3.1 Detect Missing Values
Missing values reduce data quality and can affect model performance.

In [ ]:
df.isna().sum()

The output shows whether any column contains missing values.
If all values are zero, the dataset is complete.
If any column contains missing values, we must handle them.

### 3.2 Demonstration: Introduce Artificial Missing Values
### Why?

Since our dataset has very few missing values, we introduce artificial ones *for learning purposes*.

we will be running this line:

`df_missing.loc[0:5, 'length'] = np.nan`

- `df_missing`: The pandas DataFrame you are modifying.
- `.loc[0:5, 'length']`: Selects rows 0 to 5 (inclusive) in the `length` column.
- `= np.nan`: Assigns missing values to those cells.

In [ ]:
df_missing = df.copy()
df_missing.loc[0:5, 'length'] = np.nan
df_missing.isna().sum()

In [ ]:
print("Original shape: ", df.shape)
print("After removing some values: ", df_missing.shape)

In [ ]:
df_missing.head(10)

### Strategy 1: Remove Records
This strategy removes records containing missing data.
It works well if the number of missing rows is small.

In [ ]:
df_removed = df_missing.dropna()
df_removed.shape

In [ ]:
df_removed.isna().sum()

The dataset now has fewer rows.
If only a small portion of data was missing, this method is acceptable.

However, removing too many rows can reduce model performance.

### Strategy 2: Mean Imputation

The mean represents the average value.
It is commonly used for normally distributed data.

In [ ]:
df_missing.head(10)

In [ ]:
df_imputed_mean = df_missing.copy()
df_imputed_mean['length'].fillna(df_imputed_mean['length'].mean(), inplace=True)

df_imputed_mean.isna().sum()

In [ ]:
df_imputed_mean.head(10)

Missing values are now replaced with the average password length.
This preserves dataset size but may reduce variability.
Mean imputation is sensitive to outliers.

### Strategy 3: Median Imputation

The median is more robust to outliers than the mean.
It is preferred for skewed data.

In [ ]:
df_imputed_median = df_missing.copy()
df_imputed_median['length'].fillna(df_imputed_median['length'].median(), inplace=True)

df_imputed_median.isna().sum()

In [ ]:
df_imputed_median.head(10)

Missing values are replaced with the middle value.
This approach is safer when data contains extreme values.

## 4. Handling Outliers
Outliers are extreme values that can distort models.
We will detect outliers using the IQR method on the `length` feature.

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x=df['length'])
plt.title("Boxplot of Password Length")
plt.show()

Points outside the whiskers represent potential outliers.
These extreme password lengths may influence model predictions.

### Detect Outliers using IQR
**Method: Interquartile Range (IQR)**

The IQR method defines outliers as values outside:

`Q1 - 1.5×IQR`  and  `Q3 + 1.5×IQR`

In [ ]:
Q1 = df['length'].quantile(0.25)
Q3 = df['length'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df['length'] < lower) | (df['length'] > upper)]
outliers.head(15)

The output displays passwords with extreme lengths based on statistical boundaries.
These may be very short weak passwords or unusually long strong passwords.

### Remove Outliers
We remove values outside the acceptable range.

In [ ]:
df_no_outliers = df[(df['length'] >= lower) & (df['length'] <= upper)]
print("Original shape: ", df.shape)
print("After removing outliers: ", df_no_outliers.shape)

The dataset size is slightly reduced.
Removing outliers reduces distortion but may also remove important rare cases.

#### Important Note on Removing Outliers

Not all outliers are errors.

Some extreme values may represent rare but important real-world cases.  
For example, in a password dataset, a very long password might represent a highly secure passphrase used in enterprise systems.  

If we remove such values blindly, we may lose valuable information and bias the analysis.

Before removing outliers, we should always ask:
- Is this value a data entry mistake?
- Or is it a valid but rare observation?

### Capping Outliers (Percentile Method)
Instead of removing outliers, we replace extreme values with percentile limits.

In [ ]:
lower_cap = df['length'].quantile(0.05)
upper_cap = df['length'].quantile(0.95)

df_capped = df.copy()
df_capped['length'] = df_capped['length'].clip(lower_cap, upper_cap)

## 5. Data Transformation – Normalization
Normalization scales numerical features to a similar range.
This ensures that no feature influences the model simply because it has larger numerical values.

### Min-Max Normalization
Min-Max normalization rescales numerical values to a fixed range, usually between 0 and 1.

Formula: `(x - min) / (max - min)`

This method preserves the original distribution shape and relative ordering of values.

Min-Max normalization is especially useful for distance-based models such as:
- K-Nearest Neighbors (KNN)
- K-Means clustering
- Support Vector Machines (SVM)

In [ ]:
df[['length', 'num_digits']].head()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df_scaled = df[['length', 'num_digits']].copy()

df_scaled[['length', 'num_digits']] = scaler.fit_transform(df_scaled)

df_scaled.head()

After applying Min-Max normalization, all numerical values are scaled to the range between 0 and 1.

The smallest value in each feature becomes 0, and the largest becomes 1.
All other values are proportionally mapped between these two limits.

Importantly, normalization does NOT change the relative relationships between data points.

### Z-Score Normalization
Z-score standardization transforms the data so that the mean becomes 0 and the standard deviation becomes 1.

Formula: `(x - mean) / std`

In [ ]:
from sklearn.preprocessing import StandardScaler

std_scaler = StandardScaler()
df_standardized = df[['length', 'num_digits']].copy()

df_standardized[['length', 'num_digits']] = std_scaler.fit_transform(df_standardized)

df_standardized.head()

After standardization, the numerical features are centered around 0.
Values above the original mean become **positive**, and values below the mean become **negative**.

This transformation is especially useful for:
- Linear regression
- Support Vector Machines (SVM)
- PCA

## Check Correlation Before Applying PCA

We will check whether numerical features are correlated. If features are strongly correlated, they contain overlapping information.

- **Correlation close to 1**  → Strong positive linear relationship  
- **Correlation close to -1** → Strong negative linear relationship  
- **Correlation close to 0**  → Weak or no linear relationship  

In [ ]:
plt.figure(figsize=(6,4))
sns.heatmap(df_standardized[['length','num_digits']].corr(), 
            annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap (Before PCA)")
plt.show()

The heatmap shows the correlation between `length` and `num_digits`.

- The diagonal values are 1 because each feature is perfectly correlated with itself.
- If the correlation between `length` and `num_digits` is close to 0, the two features have little linear relationship.

Since PCA is most useful when features are strongly correlated, applying PCA here is mainly for **demonstration purposes**.

## 6. Data Reduction – Principal Component Analysis (PCA)

Principal Component Analysis (PCA) is a dimensionality reduction technique.

Instead of working directly with the original features, PCA creates new features called **principal components**.

These components:
- Are linear combinations of the original features
- Are uncorrelated with each other
- Capture variance in descending order (from most important to least)

In [ ]:
from sklearn.decomposition import PCA

X = df_standardized[['length', 'num_digits']]

pca = PCA(n_components=2)
principal_components = pca.fit_transform(X)

print("Explained Variance Ratio:", pca.explained_variance_ratio_)

The `Explained Variance Ratio` indicates how much of the total information (variance) is captured by each principal component.

- If PC1 explains most of the variance, one new feature already summarizes most of the dataset's information.
- If PC1 and PC2 together explain nearly 100%, then very little information is lost.

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(principal_components[:,0], principal_components[:,1], alpha=0.1, s=1)
plt.title("PCA Projection")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.show()

Each point in this plot represents one password.

The axes no longer represent the original features (`length` and `num_digits`).
Instead:
- The horizontal axis represents Principal Component 1 (PC1).
- The vertical axis represents Principal Component 2 (PC2).

PC1 captures the direction of maximum variance in the data.
This projection allows us to visualize high-dimensional data in a lower-dimensional space.

# Assignment

In this assignment, you will:
- **Task 1**
Identify data quality issues in the dataset.

- **Task 2**
Apply one missing value strategy and explain why.

- **Task 3**
Detect and handle outliers using IQR.

- **Task 4**
Normalize numerical features using both Min-Max and Z-score.

- **Task 5**
Apply PCA and interpret explained variance.


End of lab 4.